# Read Table -> Establish Generic Silver-level Table

In [0]:
#Imports to run API Data Scraping script

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from datetime import datetime
import logging
import traceback
import json
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *


In [0]:
#parameters that need to be changed between notebooks

silver_level_schema = StructType([
    StructField('client_id', IntegerType(), True),
    StructField('current_age', IntegerType(), True),
    StructField('retirement_age', IntegerType(), True),
    StructField('birth_year', IntegerType(), True),
    StructField('birth_month', IntegerType(), True),
    StructField('gender', StringType(), True),
    StructField('address', StringType(), True),
    StructField('latitude', FloatType(), True),
    StructField('longitude', FloatType(), True),
    StructField('per_capita_income', IntegerType(), True),
    StructField('yearly_income', IntegerType(), True),
    StructField('total_debt', IntegerType(), True),
    StructField('credit_score', IntegerType(), True),
    StructField('num_credit_cards', IntegerType(), True)
])

schema = silver_level_schema
schema_path = 'financial_transactions_dataset.default'
volume_path = '/Volumes/financial_transactions_dataset/default/financial_transactions_dataset_analytics_volume'
volume_identifier = 'cards_data'
silver_table_name = 'silver_users_data'
bronze_table_name = 'bronze_users_data'

In [0]:
#Set up logger to track pipeline activities

def setup_logging():

    # Ensure logs directory exists before starting logging
    os.makedirs('logs', exist_ok=True)

    logging.basicConfig(
        level = logging.INFO,
        format = '%(asctime)s - %(levelname)s - %(message)s',
        handlers = [logging.FileHandler(f'logs/silver_level_table_run_{datetime.now().strftime('%Y%m%d')}.log'),
                    logging.StreamHandler(sys.stdout)
                   ]
    )

    return logging.getLogger(__name__)

In [0]:
def dollar_fields_cleaned(logger, df):
    logger.info(f'Converting per capita income to integer')

    df = df.withColumn('per_capita_income', F.regexp_replace(F.col('per_capita_income'), '\\$', ''))
    df = df.withColumn('per_capita_income', F.col('per_capita_income').cast('int'))

    logger.info(f'Converting yearly income to integer')

    df = df.withColumn('yearly_income', F.regexp_replace(F.col('yearly_income'), '\\$', ''))
    df = df.withColumn('yearly_income', F.col('yearly_income').cast('int'))

    logger.info(f'Converting total debt to integer')

    df = df.withColumn('total_debt', F.regexp_replace(F.col('total_debt'), '\\$', ''))
    df = df.withColumn('total_debt', F.col('total_debt').cast('int'))

    return df


In [0]:
def convert_fields_to_decimal(logger, df):
    logger.info(f'Converting latitude to decimal')

    df = df.withColumn('latitude', F.col("latitude").cast('decimal(18,2)'))

    logger.info(f'Converting longitude to decimal')

    df = df.withColumn('longitude', F.col("longitude").cast('decimal(18,2)'))

    return df

In [0]:
def convert_fields_to_integer(logger, df):
    logger.info(f'Converting fields to integer')

    df = df.withColumn('id', F.col("id").cast('int'))
    df = df.withColumn('current_age', F.col("current_age").cast('int'))
    df = df.withColumn('retirement_age', F.col("retirement_age").cast('int'))
    df = df.withColumn('birth_year', F.col("birth_year").cast('int'))
    df = df.withColumn('birth_month', F.col("birth_month").cast('int'))
    df = df.withColumn('credit_score', F.col("credit_score").cast('int'))
    df = df.withColumn('num_credit_cards', F.col("num_credit_cards").cast('int'))

    return df

In [0]:
def rename_field(logger, df):
    logger.info(f'Renaming id field to client_id.')

    df = df.withColumnRenamed('id', 'client_id')

    return df

In [0]:
#Check if a table exists in a schema; create an empty table if it does not exist.

def check_or_create_table(logger, silver_table_name, schema_path):

    logger.info(f'Checking to see if {silver_table_name} exists.')
    
    if spark.catalog.tableExists(f'{schema_path}.{silver_table_name}'):
        logger.info("Table exists; table creation skipped.")
        flag = 1
    else:
        logger.info('Table does not exist; table will be created.')
        flag = 0
    
    return flag


In [0]:
#Pull data from bronze-level preliminary table

    #Function to read a SQL data table as a dataframe

def read_table_as_df(logger, schema_path, bronze_table_name):

    logger.info(f'Reading data from {schema_path}.{bronze_table_name}')

    df = spark.read.table(f'{schema_path}.{bronze_table_name}')
    total_row_count = df.count()

    logger.info(f'Read {total_row_count} rows from {schema_path}.{bronze_table_name}')
    
    return df


In [0]:
#Function to overwrite/append data to silver level table

def create_or_append_to_table(logger, table_exists, df, schema_path, silver_table_name):

    if table_exists == 1:
        logger.info(f'Appending data to {schema_path}')

        df.write.format('delta').mode('append').saveAsTable(f'{schema_path}.{silver_table_name}')

        total_row_count = df.count()

        logger.info(f'Wrote {total_row_count} rows to {schema_path}')
    
    else:

        logger.info(f'Creating table to {schema_path}')

        df.write.format('delta').mode('overwrite').saveAsTable(f'{schema_path}.{silver_table_name}')

        total_row_count = df.count()

        logger.info(f'Wrote {total_row_count} rows to {schema_path}')
    
    return


In [0]:
def silver_level_transformation(silver_level_schema, schema_path, bronze_table_name, silver_table_name, volume_path, volume_identifier):

    schema = silver_level_schema
    schema_path = schema_path
    volume_path = volume_path
    volume_identifier = volume_identifier
    silver_table_name = silver_table_name
    bronze_table_name = bronze_table_name
    
    logger = setup_logging()

    logger.info(f'Initializing ETL on bronze dataframe')

    is_created = check_or_create_table(logger, silver_table_name, schema_path)
    bronze_df = read_table_as_df(logger, schema_path, bronze_table_name)
    dollar_df = dollar_fields_cleaned(logger, bronze_df)
    decimal_df = convert_fields_to_decimal(logger, dollar_df)
    integer_df = convert_fields_to_integer(logger, decimal_df)
    silver_df = rename_field(logger, integer_df)

    create_or_append_to_table(logger, is_created, silver_df, schema_path, silver_table_name)
    
    logger.info(f'Silver level table {silver_table_name} created or appended to successfully.')

    return


In [0]:
silver_level_transformation(silver_level_schema, schema_path, bronze_table_name, silver_table_name, volume_path, volume_identifier)